# Fine-tune Laya as the ERPNext chatbot router (TEXT_TO_SQL vs RAG)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Then Runtime → Run all. Cell 2 asks you to upload `colab_upload.zip` (in your Laya project folder).

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU - set Runtime > Change runtime type > T4 GPU, then rerun."
print("GPU:", torch.cuda.get_device_name(0))

## 2. Install Laya and upload the data

In [ ]:
!pip -q install laya==0.4.2

import os
from google.colab import files

if not os.path.exists("laya_data/train.jsonl"):
    uploaded = files.upload()   # choose colab_upload.zip
    !unzip -o -q colab_upload.zip
!ls -la laya_data
!wc -l laya_data/*.jsonl

## 3. Dry run: every row should parse (`skipped: {}`)

In [ ]:
!python -m laya.train_cli --data laya_data/train.jsonl --eval laya_data/val.jsonl     --base convaiinnovations/laya --shuffle-options --loss soft-ce --dry-run

## 4. Full fine-tune

Trains the whole model (no `--freeze-encoder`). At the end the trainer prints base vs fine-tuned accuracy on `val.jsonl`.

In [ ]:
EPOCHS = 4
OUT = "checkpoints/laya-erp-router"

!python -m laya.train_cli --data laya_data/train.jsonl --eval laya_data/val.jsonl     --base convaiinnovations/laya --out {OUT}     --shuffle-options --loss soft-ce --epochs {EPOCHS}

## 5. Evaluate: base vs fine-tuned

`test.jsonl` holds templates, doctypes and terms that are **not** in training. `handwritten_eval.jsonl` has 100 realistic questions.
Baseline (base model): **62.5%** test, **64%** handwritten.

In [ ]:
!python eval_router.py --models convaiinnovations/laya {OUT} --show-errors 15

## 6. Try your own questions

In [ ]:
import laya
from router_question import QUESTIONS, make_state

agent = laya.load(OUT)

def route(q):
    a = agent.predict(make_state(q), QUESTIONS)["answers"]["route"]
    return a["choice"], a["confidence"], a["probabilities"]

for q in [
    "What is my material receipt amount?",
    "What is a material receipt?",
    "Show me the last 10 material receipts",
    "How do I cancel a submitted sales invoice?",
    "How many purchase orders are pending approval?",
    "What does outstanding amount mean?",
]:
    choice, conf, probs = route(q)
    print(f"{choice:12} conf={conf:.2f}  {q}")

## 7. Download the trained model

Unzip it on your PC into `checkpoints/laya-erp-router/` inside the Laya project folder, so that `checkpoints/laya-erp-router/rl_agent_config.json` exists. `python test.py` will then use it automatically.


In [ ]:
!cd checkpoints && zip -q -r laya-erp-router.zip laya-erp-router
!ls -lh checkpoints/laya-erp-router.zip
files.download("checkpoints/laya-erp-router.zip")